# Kumatage: the glitter path seen from a ship

The *kumatage* was defined in 1841 as "a bright appearance in the horizon, under
the sun or moon, arising from the reflected light of those bodies from the small
rippling waves" (see [History: Kumatage](../history/kumatage.rst)). This
tutorial simulates it with the Cox-Munk model, as seen by an observer on the deck
of a ship looking toward a low Sun, then as seen from a satellite.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from coxmunk import sunglint

## Geometry of the observer

The observer looks at the sea surface with a depression angle $\delta$ below the
horizon and an azimuth offset $\alpha$ from the Sun. Seen from the observed
point of the surface, the observer is at the zenith angle $\theta_v = 90° - \delta$
and in the azimuth opposite to the Sun, $\phi = 180° + \alpha$. For a flat sea,
the image of the Sun would be a single point at $\delta = $ Sun elevation and
$\alpha = 0$.

In [ ]:
sun_elevation = 10.                    # deg above the horizon
sza = 90. - sun_elevation
delta = np.linspace(0.25, 30, 80)      # depression angle below the horizon (deg)
alpha = np.linspace(-30, 30, 81)       # azimuth offset from the Sun (deg)


def deck_view(ws, wazi=0, stats='bh2006', shadow=True):
    '''Glint reflectance I seen from a ship deck, shape (len(delta), len(alpha)).'''
    I = np.empty((len(delta), len(alpha)))
    for i, d in enumerate(delta):
        for j, a in enumerate(alpha):
            I[i, j] = sunglint(sza, 90. - d, 180. + a).sunglint(ws, wazi, stats=stats, shadow=shadow)[0]
    return I

## Kumatage and wind speed

The glitter path stretches between the horizon and the observer, below the Sun.
For a calm sea, it is narrow and concentrated around the image of the Sun; the
rougher the sea, the longer and wider the path, as more facets are tilted
enough to send the sunlight toward the observer.

The reflectance increases toward the horizon: seen at grazing angles, a given
area of the sea surface is compressed into a small solid angle (factor
$1/\mu_v$), hence the "bright appearance in the horizon" of the 1841
definition.

In [ ]:
winds = (1, 5, 12)
fig, axs = plt.subplots(1, len(winds), figsize=(18, 5.5), sharey=True)
norm = LogNorm(1e-2, 1e4)
for ax, ws in zip(axs, winds):
    I = deck_view(ws)
    im = ax.pcolormesh(alpha, -delta, np.clip(I, 1e-6, None), norm=norm, cmap='inferno', shading='auto')
    ax.plot(0, sun_elevation, 'o', ms=14, color='gold')
    ax.axhline(0, color='w', lw=0.8)
    ax.set_facecolor('#4a6fa5')
    ax.set_ylim(-30, 15)
    ax.set_title(f'wind speed {ws} m/s')
    ax.set_xlabel('azimuth from the Sun (deg)')
axs[0].set_ylabel('elevation (deg)')
fig.colorbar(im, ax=axs, label='glint reflectance I', shrink=0.9)
plt.show()

## Shadowing near the horizon

Close to the horizon, the waves hide each other: part of the facets that would
reflect the Sun toward the observer are hidden from the Sun or from the
observer. The shadowing correction reduces the glint at grazing angles, i.e.,
in the most distant part of the kumatage.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for ws in winds:
    on = [sunglint(sza, 90. - d, 180.).sunglint(ws, 0, stats='bh2006', shadow=True)[0] for d in delta]
    off = [sunglint(sza, 90. - d, 180.).sunglint(ws, 0, stats='bh2006', shadow=False)[0] for d in delta]
    line, = ax.semilogy(delta, on, label=f'{ws} m/s')
    ax.semilogy(delta, off, '--', color=line.get_color())
ax.set_xlabel('depression angle below the horizon (deg)')
ax.set_ylabel('glint reflectance I')
ax.set_title('Below the Sun (solid: with shadowing, dashed: without)')
ax.legend()
plt.show()

## From kumatage to sunglint

Seen from a satellite, the same reflection becomes the sunglint. For a sensor
looking down at moderate viewing angles, the glint is generally much brighter
than the light leaving the water (a reflectance of the order of
$10^{-3}$ to $10^{-2}$ in the visible): for ocean color, it is a noise. But its
shape depends on the wind: it is also a signal on the sea surface (see
[Wind speed from sunglint](../wind_speed.rst)).

In [ ]:
vza = np.linspace(0, 60, 121)
fig, ax = plt.subplots(figsize=(8, 5))
for ws in (2, 6, 12):
    I = [sunglint(30, v, 180).sunglint(ws, 0, stats='bh2006')[0] for v in vza]
    ax.semilogy(vza, I, label=f'{ws} m/s')
ax.axhspan(1e-3, 1e-2, color='seagreen', alpha=0.2, label='water-leaving reflectance\n(order of magnitude)')
ax.set_xlabel('viewing zenith angle (deg), principal plane')
ax.set_ylabel('glint reflectance I')
ax.set_ylim(1e-4, 1)
ax.set_title('Sun zenith angle 30°')
ax.legend()
plt.show()

## Polarization of the kumatage

The glint is polarized by the Fresnel reflection (see
[Polarized sunglint](../polarization.rst)). The Stokes vector $[I, Q, U, V]$ is
defined with respect to the meridian plane of the observer, i.e., the vertical
plane containing the line of sight. The degree of linear polarization and the
angle of polarization are

$$\mathrm{DoLP} = \frac{\sqrt{Q^2 + U^2}}{I}, \qquad \chi = \frac{1}{2} \arctan\frac{U}{Q}.$$

The same views as above are computed for the full Stokes vector; directions
with a negligible glint ($I < 10^{-2}$) are masked.

In [ ]:
def deck_stokes(ws, wazi=0, stats='bh2006', shadow=True):
    '''Glint Stokes vector [I, Q, U, V] seen from a ship deck, shape (4, len(delta), len(alpha)).'''
    S = np.empty((4, len(delta), len(alpha)))
    for i, d in enumerate(delta):
        for j, a in enumerate(alpha):
            S[:, i, j] = sunglint(sza, 90. - d, 180. + a).sunglint(ws, wazi, stats=stats, shadow=shadow)
    return S


stokes = {ws: deck_stokes(ws) for ws in winds}

### Degree of linear polarization

The DoLP depends on the angle of incidence on the facets only, not on the wind
speed: the wind changes the extent of the glitter path, not its polarization.
With the Sun 10° above the horizon, the scattering angle is far from the
Brewster angle near the image of the Sun (grazing incidence) and the glint is
weakly polarized; the DoLP increases toward the observer, as the scattering
angle decreases toward its Brewster value (about 74°).

In [ ]:
fig, axs = plt.subplots(1, len(winds), figsize=(18, 5.5), sharey=True)
for ax, ws in zip(axs, winds):
    I, Q, U, V = stokes[ws]
    dolp = np.where(I > 1e-2, np.hypot(Q, U) / I, np.nan)
    im = ax.pcolormesh(alpha, -delta, dolp, vmin=0, vmax=1, cmap='viridis', shading='auto')
    ax.plot(0, sun_elevation, 'o', ms=14, color='gold')
    ax.axhline(0, color='w', lw=0.8)
    ax.set_facecolor('#4a6fa5')
    ax.set_ylim(-30, 15)
    ax.set_title(f'wind speed {ws} m/s')
    ax.set_xlabel('azimuth from the Sun (deg)')
axs[0].set_ylabel('elevation (deg)')
fig.colorbar(im, ax=axs, label='glint degree of linear polarization', shrink=0.9)
plt.show()

### Angle of polarization

The angle of polarization $\chi$ is counted from the meridian (vertical) plane
of the observer. Below the Sun ($\alpha = 0$), the plane of incidence is
vertical and the reflected light is polarized horizontally ($\chi = \pm 90°$):
this is why polarized sunglasses, which block horizontally polarized light,
dim the glitter path. Off the Sun's vertical, the plane of incidence on the
facets tilts and the polarization rotates symmetrically on each side. Like the
DoLP, $\chi$ is a purely geometrical quantity: the wind changes where it can
be seen, not its value. A cyclic colormap is used since $\chi = -90°$ and
$\chi = +90°$ are the same direction. The sea outside the glitter path is shown in gray.

In [ ]:
fig, axs = plt.subplots(1, len(winds), figsize=(18, 5.5), sharey=True)
for ax, ws in zip(axs, winds):
    I, Q, U, V = stokes[ws]
    aolp = np.where(I > 1e-2, np.degrees(0.5 * np.arctan2(U, Q)), np.nan)
    im = ax.pcolormesh(alpha, -delta, aolp, vmin=-90, vmax=90, cmap='twilight', shading='auto')
    ax.plot(0, sun_elevation, 'o', ms=14, color='gold')
    ax.axhline(0, color='w', lw=0.8)
    ax.set_facecolor('0.6')
    ax.set_ylim(-30, 15)
    ax.set_title(f'wind speed {ws} m/s')
    ax.set_xlabel('azimuth from the Sun (deg)')
axs[0].set_ylabel('elevation (deg)')
cb = fig.colorbar(im, ax=axs, label='glint angle of polarization $\\chi$ (deg)', shrink=0.9)
cb.set_ticks([-90, -45, 0, 45, 90])
plt.show()